In [3]:
# ==============================
# Upload Audio File for Prediction
# ==============================

from google.colab import files
import librosa
import numpy as np
import pandas as pd # Needed for feature dataframe
from joblib import load
import os # Needed for os.path.join
from google.colab import drive # Added for drive mount
import warnings
import librosa


# Mount Google Drive to access models and other files
drive.mount('/content/drive')
print('✅ Google Drive mounted!')

print("Upload an audio (.wav) file for prediction:")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
print("Uploaded file:", file_name)


# ==============================
# Define Constants and Helpers (Copied from earlier cells)
# ==============================

SR = 16000  # Sampling rate (16,000 Hz as in paper)
SEGMENT_LEN = 4 * SR  # 4-second segments

def truncate_or_pad(y, length=SEGMENT_LEN):
    """Ensure all segments are exactly 4 seconds."""
    if len(y) > length:
        return y[:length]
    elif len(y) < length:
        return np.pad(y, (0, length - len(y)))
    return y

# ─────────────────────────────────────────────────────────
# FEATURE EXTRACTION FUNCTION (Copied from Section 3)
# ─────────────────────────────────────────────────────────
def extract_80_features(y, sr=SR):
    """
    Extract all 80 features as described in Table 4 of the paper.
    Returns a flat feature vector with named columns.
    """
    features = {}

    # ── Chroma CENS (3 features) ──────────────────────────
    chroma_cens = librosa.feature.chroma_cens(y=y, sr=sr)
    features['chroma_cens_mean'] = np.mean(chroma_cens)
    features['chroma_cens_std']  = np.std(chroma_cens)
    features['chroma_cens_var']  = np.var(chroma_cens)

    # ── Mel-Spectrogram (3 features) ──────────────────────
    mel = librosa.feature.melspectrogram(y=y, sr=sr)
    features['mel_mean'] = np.mean(mel)
    features['mel_std']  = np.std(mel)
    features['mel_var']  = np.var(mel)

    # ── MFCC global (6 features) ──────────────────────────
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    features['mfcc_mean'] = np.mean(mfcc)
    features['mfcc_std']  = np.std(mfcc)
    features['mfcc_var']  = np.var(mfcc)

    mfcc_delta = librosa.feature.delta(mfcc)
    features['mfcc_delta_mean'] = np.mean(mfcc_delta)
    features['mfcc_delta_std']  = np.std(mfcc_delta)
    features['mfcc_delta_var']  = np.var(mfcc_delta)

    # ── Spectral Centroid (3 features) ────────────────────
    cent = librosa.feature.spectral_centroid(y=y, sr=sr)
    features['cent_mean'] = np.mean(cent)
    features['cent_std']  = np.std(cent)
    features['cent_var']  = np.var(cent)

    # ── Spectral Bandwidth (3 features) ───────────────────
    spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    features['spec_bw_mean'] = np.mean(spec_bw)
    features['spec_bw_std']  = np.std(spec_bw)
    features['spec_bw_var']  = np.var(spec_bw)

    # ── Spectral Roll-off (3 features) ────────────────────
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    features['rolloff_mean'] = np.mean(rolloff)
    features['rolloff_std']  = np.std(rolloff)
    features['rolloff_var']  = np.var(rolloff)

    # ── Zero-Crossing Rate (3 features) ───────────────────
    zcr = librosa.feature.zero_crossing_rate(y)
    features['zcr_mean'] = np.mean(zcr)
    features['zcr_std']  = np.std(zcr)
    features['zcr_var']  = np.var(zcr)

    # ── Harmonic / Percussive (6 features) ────────────────
    harm, perc = librosa.effects.hpss(y)
    features['harm_mean'] = np.mean(harm)
    features['harm_std']  = np.std(harm)
    features['harm_var']  = np.var(harm)
    features['perc_mean'] = np.mean(perc)
    features['perc_std']  = np.std(perc)
    features['perc_var']  = np.var(perc)

    # ── MFCC per-coefficient mean (13 features) ───────────
    # Eq.(5) in paper
    for i in range(13):
        features[f'mfccs_mean_{i}'] = np.mean(mfcc[i])

    # ── MFCC per-coefficient std (13 features) ────────────
    # Eq.(6) & (7) in paper
    for i in range(13):
        features[f'mfccs_std_{i}'] = np.std(mfcc[i])

    # ── Chroma per-bin mean (12 features) ─────────────────
    # Eq.(8) in paper
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    for i in range(12):
        features[f'chroma_mean_{i}'] = np.mean(chroma[i])

    # ── Chroma per-bin std (12 features) ──────────────────
    for i in range(12):
        features[f'chroma_std_{i}'] = np.std(chroma[i])

    return features

# ─────────────────────────────────────────────────────────
# BEST FEATURES FOR THE MODEL (Copied from Section 9/10)
# ─────────────────────────────────────────────────────────
mfcc_mean_cols  = [f'mfccs_mean_{i}' for i in range(13)]
mfcc_std_cols   = [f'mfccs_std_{i}'  for i in range(13)]
MFCC_BEST_COLS = mfcc_mean_cols + mfcc_std_cols # This matches the best model's training features.


# ==============================
# Load Trained Model & Scaler
# ==============================

# Define MODEL_DIR again for clarity. This variable holds the path to your saved models.
MODEL_DIR = '/content/drive/MyDrive/respiratory_disease_classification-master/data/outputs/models'

model = load(os.path.join(MODEL_DIR, 'best_knn_mfcc_mean_std.joblib'))
scaler = load(os.path.join(MODEL_DIR, 'scaler.joblib'))
label_encoder = load(os.path.join(MODEL_DIR, 'label_encoder.joblib'))

print("✅ Model, Scaler, and Label Encoder loaded successfully!")


# ==============================
# Process Uploaded File and Predict
# ==============================

# Load audio with correct SR and apply truncation/padding
audio, _ = librosa.load(file_name, sr=SR)
audio = truncate_or_pad(audio)

# Extract the 80 features using the correct function
raw_features_dict = extract_80_features(audio, sr=SR)

# Convert to DataFrame to ensure correct column order and selection
# The scaler was fitted on the full 80 features (`X_all_scaled`)
all_80_features_df = pd.DataFrame([raw_features_dict])

# Scale ALL 80 features first
features_scaled_all = scaler.transform(all_80_features_df)
features_scaled_all_df = pd.DataFrame(features_scaled_all, columns=all_80_features_df.columns)

# Then select only the features the best model was trained on (`MFCC_BEST_COLS`)
features_for_prediction = features_scaled_all_df[MFCC_BEST_COLS]

# ==============================
# Prediction
# ==============================

prediction = model.predict(features_for_prediction)

# Decode label if encoder used
try:
    prediction_label = label_encoder.inverse_transform(prediction)
    print("Predicted Class:", prediction_label[0])
except Exception as e:
    print(f"Error during label decoding: {e}")
    print("Raw Prediction:", prediction[0])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted!
Upload an audio (.wav) file for prediction:


Saving test1.wav to test1 (1).wav
Uploaded file: test1 (1).wav
✅ Model, Scaler, and Label Encoder loaded successfully!
Predicted Class: Healthy
